# Évaluation intermédiaire Python pour la Data Science
**Exploitation de données électorales avec Python**

In [19]:
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 4.7 MB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.2/607.2 kB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [great_tables] [great_tables]
Note: you may need to restart the kernel to use updated packages.


In [28]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from great_tables import GT


# Optionnel : si vous souhaitez de beaux tableaux comme demandé à la Q3
# !pip install great_tables
# from great_tables import GT 

# Chargement des données
url = 'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
# On force le type string pour les codes afin d'éviter la perte des zéros initiaux
df = pd.read_csv(url, dtype={'code_departement': str, 'code_commune': str})

# Aperçu des données
display(df.head())

/tmp/ipykernel_12965/3790486426.py:14: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url, dtype={'code_departement': str, 'code_commune': str})


,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix
0,01,Ain,001,L'Abergement-Clémenciat,Nathalie,ARTHAUD,3
1,01,Ain,002,L'Abergement-de-Varey,Nathalie,ARTHAUD,2
2,01,Ain,004,Ambérieu-en-Bugey,Nathalie,ARTHAUD,38
3,01,Ain,005,Ambérieux-en-Dombes,Nathalie,ARTHAUD,8
4,01,Ain,006,Ambléon,Nathalie,ARTHAUD,0


## 1. Explorations générales
### Question 1
Création et mise à jour des variables `code_commune` et `candidat`.

In [29]:
# Mise à jour du code_commune (Département + Code commune sur 3 caractères)
# On s'assure que le code commune interne a bien 3 chiffres (ex: 1 devient 001)
df['code_commune_format'] = df['code_commune'].astype(str).str.zfill(3)
df['code_commune'] = df['code_departement'].astype(str) + df['code_commune_format']
df = df.drop(columns=['code_commune_format'])

# Création de la colonne candidat (Prénom + Nom)
# On remplit les valeurs manquantes éventuelles par du vide pour éviter les erreurs
df['candidat'] = df['prenom'].fillna('') + ' ' + df['nom'].fillna('')
# On retire les espaces superflus
df['candidat'] = df['candidat'].str.strip()

display(df[['code_departement', 'code_commune', 'candidat', 'voix']].head())

,code_departement,code_commune,candidat,voix
0,01,01001,Nathalie ARTHAUD,3
1,01,01002,Nathalie ARTHAUD,2
2,01,01004,Nathalie ARTHAUD,38
3,01,01005,Nathalie ARTHAUD,8
4,01,01006,Nathalie ARTHAUD,0


In [30]:
# Essai pour Montrouge
df[df['libelle_commune'] == 'Montrouge']

,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix,candidat
34535,92,Hauts-de-Seine,92049,Montrouge,Nathalie,ARTHAUD,96,Nathalie ARTHAUD
69780,92,Hauts-de-Seine,92049,Montrouge,Fabien,ROUSSEL,499,Fabien ROUSSEL
105025,92,Hauts-de-Seine,92049,Montrouge,Emmanuel,MACRON,8291,Emmanuel MACRON
140270,92,Hauts-de-Seine,92049,Montrouge,Jean,LASSALLE,349,Jean LASSALLE
175515,92,Hauts-de-Seine,92049,Montrouge,Marine,LE PEN,1775,Marine LE PEN
210760,92,Hauts-de-Seine,92049,Montrouge,Éric,ZEMMOUR,1466,Éric ZEMMOUR
246005,92,Hauts-de-Seine,92049,Montrouge,Jean-Luc,MÉLENCHON,6683,Jean-Luc MÉLENCHON
281250,92,Hauts-de-Seine,92049,Montrouge,Anne,HIDALGO,508,Anne HIDALGO
316495,92,Hauts-de-Seine,92049,Montrouge,Yannick,JADOT,1952,Yannick JADOT
351740,92,Hauts-de-Seine,92049,Montrouge,Valérie,PÉCRESSE,1468,Valérie PÉCRESSE


### Question 2
Comptage des candidats (en excluant les votes non exprimés/abstentions).

In [31]:
# On retire les votes non exprimés et les abstentions 
non_candidats = ['BLANC', 'BLANCS', 'NUL', 'NULS', 'ABSTENTION', 'ABSTENTIONS'] # Attention au pluriel (compte 15 candidats sinon)

# On filtre la liste des candidats uniques
candidats_uniques = [
    c for c in df['candidat'].unique() 
    # On vérifie que c n'est pas vide/NaN ET que le mot exact n'est pas dans la liste d'exclusion
    if str(c).strip() and str(c).strip().upper() not in non_candidats
]

candidats = len(candidats_uniques)

# Réponse formatée demandée
print(f"En 2022, il y avait {candidats} candidats à l'élection présidentielle.")

En 2022, il y avait 12 candidats à l'élection présidentielle.


### Question 3
Calcul des scores nationaux de chaque candidat.

In [ ]:
# On ne garde que les votes exprimés (les vrais candidats)
df_exprimes = df[df['candidat'].isin(candidats_uniques)].copy()

# Total des voix exprimées
total_voix_exprimees = df_exprimes['voix'].sum()

# Groupby pour avoir la somme par candidat
national_scores = df_exprimes.groupby('candidat')['voix'].sum().reset_index()
national_scores = national_scores.rename(columns={'voix': 'votes_national'})

# Calcul du pourcentage
national_scores['score_national'] = (national_scores['votes_national'] / total_voix_exprimees) * 100

# Tri décroissant comme dans l'exemple 
national_scores = national_scores.sort_values(by='score_national', ascending=False).reset_index(drop=True)

# Formatage pour l'affichage final
national_scores_display = national_scores.copy()
national_scores_display['score_national'] = national_scores_display['score_national'].apply(lambda x: f"{x:.2f}%")


# Affichage great_tables
GT(national_scores_display).tab_header(title="Résultats du premier tour (10 avril 2022)")

GT(_tbl_data=                 candidat  votes_national score_national
0         Emmanuel MACRON         9783058         27.85%
1           Marine LE PEN         8133828         23.15%
2      Jean-Luc MÉLENCHON         7712520         21.95%
3            Éric ZEMMOUR         2485226          7.07%
4        Valérie PÉCRESSE         1679001          4.78%
5           Yannick JADOT         1627853          4.63%
6           Jean LASSALLE         1101387          3.13%
7          Fabien ROUSSEL          802422          2.28%
8   Nicolas DUPONT-AIGNAN          725176          2.06%
9            Anne HIDALGO          616478          1.75%
10        Philippe POUTOU          268904          0.77%
11       Nathalie ARTHAUD          197094          0.56%, _body=<great_tables._gt_data.Body object at 0x7fd59d5fdae0>, _boxhead=Boxhead([ColInfo(var='candidat', type=<ColInfoTypeEnum.default: 1>, column_label='candidat', column_align='left', column_width=None), ColInfo(var='votes_national', type=<ColInfoTypeEnum.default: 1>, column_label='votes_national', column_align='right', column_width=None), ColInfo(var='score_national', type=<ColInfoTypeEnum.default: 1>, column_label='score_national', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7fd59d8e3750>, _spanners=Spanners([]), _heading=Heading(title='Résultats du premier tour (10 avril 2022)', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7fd5073fc640>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7fd509bbf650>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7fd59d8e3c50>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), ta

## 2. Comparaison des scores départements aux moyennes nationales
### Question 4
Création du dataframe `score_departements`.

In [35]:
# Total des voix par candidat et par département
score_departements = df_exprimes.groupby(['code_departement', 'candidat'])['voix'].sum().reset_index()
score_departements = score_departements.rename(columns={'voix': 'votes_departement'})

# Calcul du total des votes exprimés PAR département
total_par_dept = score_departements.groupby('code_departement')['votes_departement'].sum().reset_index()
total_par_dept = total_par_dept.rename(columns={'votes_departement': 'total_dept'})

# Jointure pour calculer le score départemental
score_departements = score_departements.merge(total_par_dept, on='code_departement')
score_departements['score_departement'] = (score_departements['votes_departement'] / score_departements['total_dept']) * 100

# On nettoie la colonne temporaire
score_departements = score_departements.drop(columns=['total_dept'])

# Vérification avec l'Aude (département 11)
aude_check = score_departements[score_departements['code_departement'] == '11'].sort_values(by='score_departement', ascending=False)
display(aude_check.head())

,code_departement,candidat,votes_departement,score_departement
125,11,Marine LE PEN,64027,30.140849
121,11,Emmanuel MACRON,43104,20.291301
124,11,Jean-Luc MÉLENCHON,42039,19.789950
131,11,Éric ZEMMOUR,18434,8.677845
123,11,Jean LASSALLE,12382,5.828853


### Question 5 & 6
Jointure avec le score national et calcul de la surreprésentation.